# PharmShed Ensemble Model

**Task:** Multi-class classification — predict which of 217 pharmaceuticals a person is prescribed  
**Strategy:** Soft voting — average probability vectors from all 5 base models, pick highest probability drug  
**Base models:** XGBoost Super, RealMLP Super, KNN Super, SVM Super, TabICL  
**Input:** One `*_proba_2022.csv` file per model — shape (n_observations, 218) with Observation_ID + 217 drug probability columns  
**Evaluation:** Same metrics as base models — accuracy, macro/micro recall, macro/micro F2, drugs recalled  

## How Soft Voting Works

For each observation:
1. Each model provides a probability vector over all 217 drugs
2. The 5 probability vectors are averaged column by column
3. The drug with the highest average probability is the final prediction

This approach uses each model's confidence — not just its hard prediction — and naturally
favors drugs where multiple models agree.

## Required Input Files (place in same folder as this notebook)

| File | Source model |
|---|---|
| `xgboost_super_proba_2022.csv` | XGBoost Super (`xgboost_super_dataset.ipynb`) |
| `realmlp_super_proba_2022.csv` | RealMLP Super (`realmlp_super_dataset.ipynb`) |
| `knn_super_proba_2022.csv` | KNN Super (`knn_super_dataset.ipynb`) |
| `svm_super_proba_2022.csv` | SVM Super (`svm_super_dataset.ipynb`) |
| `tabicl_super_proba_2022.csv` | TabICL (Vanessa) |
| `xgboost_super_label_encoder.joblib` | XGBoost Super (used as reference encoder) |
| `data_2022.csv` | Ground truth labels for 2022 validation |
| `metadata_2022.csv` | Observation_ID to Person_ID mapping |

## Output Files

| File | Contents |
|---|---|
| `ensemble_cv_results.csv` | Overall validation metrics |
| `ensemble_per_drug_recall.csv` | Per-drug recall on 2022 data |
| `ensemble_proba_2022.csv` | Averaged probability vectors (for reference) |
| `ensemble_vs_base_models.csv` | Side-by-side comparison of all models |


In [ ]:
# Install required libraries.
!pip install scikit-learn permetrics joblib

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

import sklearn
print('pandas version:      ', pd.__version__)
print('numpy version:       ', np.__version__)
print('scikit-learn version:', sklearn.__version__)

# All files are in the same folder as this notebook.
DATA_DIR = './'
print(f'\nData directory: {DATA_DIR}')

In [ ]:
# Define which model probability files to load.
# Each entry: (model_name, filename)
# All files must have Observation_ID as first column + 217 drug probability columns.
# Remove or comment out any model whose proba file is not yet available.
MODEL_FILES = [
    ('XGBoost_Super', 'xgboost_super_proba_2022.csv'),
    ('RealMLP_Super', 'realmlp_super_proba_2022.csv'),
    ('KNN_Super',     'knn_super_proba_2022.csv'),
    ('SVM_Super',     'svm_super_proba_2022.csv'),
    ('TabICL',        'tabicl_super_proba_2022.csv'),
]

# Check which files are present before loading.
print('Checking for model probability files...')
available = []
missing   = []
for model_name, fname in MODEL_FILES:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        available.append((model_name, fname))
        print(f'  FOUND:   {fname}')
    else:
        missing.append((model_name, fname))
        print(f'  MISSING: {fname}')

print(f'\nAvailable: {len(available)} / {len(MODEL_FILES)} models')
if missing:
    print(f'Missing models will be excluded from ensemble: {[m for m, _ in missing]}')

# Hard stop if fewer than 2 models are available.
# Ensemble needs at least 2 models to be meaningful.
assert len(available) >= 2, \
    f'ERROR: Need at least 2 model proba files, found {len(available)}.'

In [ ]:
# Load the reference LabelEncoder.
# All models must use the same drug-to-integer mapping.
# XGBoost Super encoder is used as the reference — all other models
# must have been fitted on the same drug list in the same order.
le = joblib.load(f'{DATA_DIR}xgboost_super_label_encoder.joblib')
drug_classes = list(le.classes_)

print('Reference LabelEncoder loaded.')
print(f'Number of drug classes: {len(drug_classes)}')
print(f'Sample mapping (first 5):')
for i, drug in enumerate(drug_classes[:5]):
    print(f'  {drug} -> {i}')

assert len(drug_classes) == 217, \
    f'ERROR: Expected 217 drug classes, found {len(drug_classes)}.'

In [ ]:
# Load all available model probability files.
# Each file: Observation_ID column + 217 drug probability columns.
# Align all files on Observation_ID so rows match across models.

proba_dict = {}  # model_name -> DataFrame (index=Observation_ID, cols=drug classes)

for model_name, fname in available:
    path = os.path.join(DATA_DIR, fname)
    df   = pd.read_csv(path)

    # Validate structure
    assert 'Observation_ID' in df.columns, \
        f'ERROR: {fname} missing Observation_ID column.'

    # Check drug columns match reference encoder
    drug_cols_in_file = [c for c in df.columns if c != 'Observation_ID']
    missing_drugs     = set(drug_classes) - set(drug_cols_in_file)
    extra_drugs       = set(drug_cols_in_file) - set(drug_classes)
    assert len(missing_drugs) == 0, \
        f'ERROR: {fname} is missing drug columns: {missing_drugs}'
    assert len(extra_drugs) == 0, \
        f'ERROR: {fname} has unexpected drug columns: {extra_drugs}'

    # Set index to Observation_ID and reorder columns to match reference
    df = df.set_index('Observation_ID')[drug_classes]
    proba_dict[model_name] = df

    print(f'Loaded {model_name}: shape {df.shape}')

# Verify all models have the same Observation_IDs
obs_ids = [set(df.index) for df in proba_dict.values()]
common_obs = obs_ids[0].intersection(*obs_ids[1:])
print(f'\nCommon Observation_IDs across all models: {len(common_obs):,}')

# Warn if any models have different observation sets
for model_name, df in proba_dict.items():
    diff = len(set(df.index)) - len(common_obs)
    if diff > 0:
        print(f'  WARNING: {model_name} has {diff} extra Observation_IDs not in all models.')

# Align all to common observations
common_obs_sorted = sorted(common_obs)
for model_name in proba_dict:
    proba_dict[model_name] = proba_dict[model_name].loc[common_obs_sorted]

print(f'All models aligned to {len(common_obs_sorted):,} common observations.')

In [ ]:
# Soft voting — average probability vectors across all models.
# For each observation, each model contributes an equal weight (1/n_models).
# The drug with the highest average probability is the ensemble prediction.

n_models = len(proba_dict)
print(f'Averaging probabilities across {n_models} models...')

# Stack all probability arrays and average
proba_arrays  = np.stack([df.values for df in proba_dict.values()], axis=0)
# Shape: (n_models, n_observations, 217)

avg_proba = proba_arrays.mean(axis=0)
# Shape: (n_observations, 217)

print(f'Averaged probability matrix shape: {avg_proba.shape}')
print(f'Row sums (should all be ~1.0): min={avg_proba.sum(axis=1).min():.4f}, '
      f'max={avg_proba.sum(axis=1).max():.4f}')

# Hard predictions — argmax of averaged probabilities
y_pred_ensemble = avg_proba.argmax(axis=1)

# Save averaged probability matrix for reference
avg_proba_df = pd.DataFrame(avg_proba, columns=drug_classes)
avg_proba_df.insert(0, 'Observation_ID', common_obs_sorted)
avg_proba_df.to_csv('ensemble_proba_2022.csv', index=False)
print(f'\nEnsemble probability output saved to ensemble_proba_2022.csv')
print(f'Shape: {avg_proba_df.shape}')

In [ ]:
# Load ground truth labels for 2022 validation.
# Align to the same Observation_IDs used by the ensemble.

data_2022 = pd.read_csv(f'{DATA_DIR}data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print('2022 data shape:', data_2022.shape)

# Filter to common Observation_IDs
data_2022 = data_2022[data_2022['Observation_ID'].isin(common_obs_sorted)].copy()
data_2022 = data_2022.set_index('Observation_ID').loc[common_obs_sorted].reset_index()
print(f'2022 rows after alignment: {len(data_2022):,}')

# Drop any drugs not seen during training
unseen = set(data_2022['Drug'].unique()) - set(le.classes_)
if unseen:
    print(f'Unseen drugs in 2022 (will be dropped): {unseen}')
    data_2022 = data_2022[data_2022['Drug'].isin(le.classes_)].reset_index(drop=True)
    print(f'2022 rows after filtering: {len(data_2022):,}')

y_2022_encoded = le.transform(data_2022['Drug'])

# Re-align ensemble predictions to filtered observations
obs_id_to_idx   = {obs_id: i for i, obs_id in enumerate(common_obs_sorted)}
filtered_idx    = [obs_id_to_idx[oid] for oid in data_2022['Observation_ID']]
y_pred_filtered = y_pred_ensemble[filtered_idx]

print(f'\nGround truth labels loaded: {len(y_2022_encoded):,} observations')
print(f'Unique drugs in 2022 ground truth: {len(set(y_2022_encoded))}')

In [ ]:
# Compute ensemble validation metrics.
# Same metric set used across all base models for direct comparison.

acc_ens   = accuracy_score(y_2022_encoded, y_pred_filtered)
kappa_ens = cohen_kappa_score(y_2022_encoded, y_pred_filtered)
mcc_ens   = matthews_corrcoef(y_2022_encoded, y_pred_filtered)

evaluator_ens     = ClassificationMetric(y_2022_encoded, y_pred_filtered)
macro_prec_ens    = evaluator_ens.precision_score(average='macro')
micro_prec_ens    = evaluator_ens.precision_score(average='micro')
macro_recall_ens  = evaluator_ens.recall_score(average='macro')
micro_recall_ens  = evaluator_ens.recall_score(average='micro')
macro_f1_ens      = evaluator_ens.f1_score(average='macro')
micro_f1_ens      = evaluator_ens.f1_score(average='micro')
macro_f2_ens      = evaluator_ens.fbeta_score(beta=2, average='macro')
micro_f2_ens      = evaluator_ens.fbeta_score(beta=2, average='micro')

print('ENSEMBLE VALIDATION — MEPS 2022 Results')
print('='*50)
print(f'Models included:   {list(proba_dict.keys())}')
print(f'Accuracy:          {acc_ens:.4f}')
print(f'Cohen Kappa:       {kappa_ens:.4f}')
print(f'MCC:               {mcc_ens:.4f}')
print(f'Macro Precision:   {macro_prec_ens:.4f}')
print(f'Micro Precision:   {micro_prec_ens:.4f}')
print(f'Macro Recall:      {macro_recall_ens:.4f}')
print(f'Micro Recall:      {micro_recall_ens:.4f}')
print(f'Macro F1:          {macro_f1_ens:.4f}')
print(f'Micro F1:          {micro_f1_ens:.4f}')
print(f'Macro F2:          {macro_f2_ens:.4f}')
print(f'Micro F2:          {micro_f2_ens:.4f}')

# Save ensemble validation summary
ensemble_summary = pd.DataFrame([{
    'model':            'Ensemble_SoftVoting',
    'models_included':  str(list(proba_dict.keys())),
    'dataset':          'MEPS_2022_internal_validation',
    'accuracy':         acc_ens,
    'cohen_kappa':      kappa_ens,
    'mcc':              mcc_ens,
    'macro_precision':  macro_prec_ens,
    'micro_precision':  micro_prec_ens,
    'macro_recall':     macro_recall_ens,
    'micro_recall':     micro_recall_ens,
    'macro_f1':         macro_f1_ens,
    'micro_f1':         micro_f1_ens,
    'macro_f2':         macro_f2_ens,
    'micro_f2':         micro_f2_ens,
}])
ensemble_summary.to_csv('ensemble_validation_summary.csv', index=False)
print('\nEnsemble validation summary saved to ensemble_validation_summary.csv')

In [ ]:
# Per-drug recall on 2022 data.
# Key output — shows which drugs the ensemble recalls vs individual models.

report_ens = classification_report(
    y_2022_encoded, y_pred_filtered,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

drug_recall_ens = pd.DataFrame([
    {
        'Drug':           drug,
        'Recall_2022':    report_ens[drug]['recall'],
        'Precision_2022': report_ens[drug]['precision'],
        'F1_2022':        report_ens[drug]['f1-score'],
        'Support_2022':   report_ens[drug]['support']
    }
    for drug in le.classes_ if drug in report_ens
]).sort_values('Recall_2022', ascending=False)

print('Top 15 drugs by recall (ensemble):')
print(drug_recall_ens.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall (ensemble):')
print(drug_recall_ens.tail(15).to_string(index=False))

# Drugs recalled summary
drugs_recalled_ens = (drug_recall_ens['Recall_2022'] > 0).sum()
print(f'\nDrugs recalled (recall > 0):    {drugs_recalled_ens} / {len(drug_recall_ens)}')
print(f'Drugs with recall >= 0.1: {(drug_recall_ens["Recall_2022"] >= 0.1).sum()}')
print(f'Drugs with recall >= 0.5: {(drug_recall_ens["Recall_2022"] >= 0.5).sum()}')

drug_recall_ens.to_csv('ensemble_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to ensemble_per_drug_recall.csv')

In [ ]:
# Compare ensemble vs all base models side by side.
# Loads each base model validation summary and combines into one table.
# This is the primary results table for the paper.

BASE_MODEL_SUMMARIES = [
    'xgboost_super_validation_summary.csv',
    'realmlp_super_validation_summary.csv',
    'knn_super_validation_summary.csv',
    'svm_super_validation_summary.csv',
    'tabicl_super_validation_summary.csv',
]

all_summaries = []
for fname in BASE_MODEL_SUMMARIES:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        all_summaries.append(df)
        print(f'Loaded: {fname}')
    else:
        print(f'Not found (skipped): {fname}')

# Add ensemble summary
all_summaries.append(ensemble_summary[[
    'model', 'dataset', 'accuracy', 'cohen_kappa', 'mcc',
    'macro_precision', 'micro_precision', 'macro_recall',
    'micro_recall', 'macro_f2', 'micro_f2'
]])

comparison_df = pd.concat(all_summaries, ignore_index=True)

print('\nFULL MODEL COMPARISON')
print('='*80)
print(comparison_df[['model', 'accuracy', 'macro_recall', 'macro_f2']].to_string(index=False))

comparison_df.to_csv('ensemble_vs_base_models.csv', index=False)
print('\nFull comparison saved to ensemble_vs_base_models.csv')

In [ ]:
# Per-drug recall comparison — ensemble vs each base model.
# Shows for each drug which model recalled it best.
# This is the most granular comparison and justifies the ensemble approach.

BASE_DRUG_RECALLS = [
    ('XGBoost_Super', 'xgboost_super_per_drug_recall.csv',  'Mean_Recall_XGBoost_Super'),
    ('RealMLP_Super', 'realmlp_super_per_drug_recall.csv',  'Mean_Recall_RealMLP_Super'),
    ('KNN_Super',     'knn_super_per_drug_recall.csv',      'Mean_Recall_KNN_Super'),
    ('SVM_Super',     'svm_super_per_drug_recall.csv',      'Mean_Recall_SVM_Super'),
    ('TabICL',        'tabicl_super_per_drug_recall.csv',   'Mean_Recall_TabICL'),
]

# Start with ensemble per-drug recall
comparison_per_drug = drug_recall_ens[['Drug', 'Recall_2022']].rename(
    columns={'Recall_2022': 'Recall_Ensemble'}
)

# Merge in each base model's per-drug recall
for model_name, fname, col_name in BASE_DRUG_RECALLS:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)[['Drug', col_name]].rename(
            columns={col_name: f'Recall_{model_name}'}
        )
        comparison_per_drug = comparison_per_drug.merge(df, on='Drug', how='left')
        print(f'Merged: {fname}')
    else:
        print(f'Not found (skipped): {fname}')

# For each drug identify which model had the best recall
recall_cols = [c for c in comparison_per_drug.columns if c.startswith('Recall_')]
comparison_per_drug['Best_Model'] = comparison_per_drug[recall_cols].idxmax(axis=1).str.replace('Recall_', '')
comparison_per_drug['Best_Recall'] = comparison_per_drug[recall_cols].max(axis=1)

# Summary — how many drugs does each model win on?
print('\nDrugs where each model has highest recall:')
print(comparison_per_drug['Best_Model'].value_counts().to_string())

comparison_per_drug = comparison_per_drug.sort_values('Recall_Ensemble', ascending=False)
comparison_per_drug.to_csv('ensemble_per_drug_comparison.csv', index=False)
print('\nPer-drug comparison saved to ensemble_per_drug_comparison.csv')

In [ ]:
# Final output file summary.
print('\n' + '='*55)
print('ALL OUTPUTS SAVED')
print('='*55)
print('  ensemble_validation_summary.csv   <- overall metrics')
print('  ensemble_per_drug_recall.csv      <- per drug recall')
print('  ensemble_proba_2022.csv           <- averaged probabilities')
print('  ensemble_vs_base_models.csv       <- model comparison table')
print('  ensemble_per_drug_comparison.csv  <- per drug best model')
print(f'\nModels included in this ensemble run: {list(proba_dict.keys())}')